In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
import torchaudio.transforms as T
import math   
from src.dataset import NSynth
import torch

mnist_datasets_path = r'C:\Users\Articuno\Desktop\TFG-info\data\mnist'
diff_mnist_model_path = r'C:\Users\Articuno\Desktop\TFG-info\data\models\diff_mnist.pth'

diff_model_path = r'C:\Users\Articuno\Desktop\TFG-info\data\models\diff.pth'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


ModuleNotFoundError: No module named 'torch'

In [ ]:
# STFT transform
sample_rate = 16000
n_fft = 1500
hop_length = 250
win_length = n_fft
stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=2.0, onesided=False, center=False
).to(device)

In [ ]:
class Scheduler(nn.Module):
    def __init__(self, num_epochs, beta_init=1e-4, beta_finish=0.02):
        super().__init__()
        self.beta = torch.linspace(beta_init, beta_finish, num_epochs, device=device)
        alpha = 1 - self.beta
        self.alpha = torch.cumprod(alpha, dim=0).requires_grad_(False)
        
    def forward(self, t):
        return self.beta[t], self.alpha[t]


In [ ]:
class Diffuser(nn.Module):
    def __init__(self, model, scheduler):
        super().__init__()
        self.model = model
        self.scheduler = scheduler

    # x: input image, t: time step
    def forward(self, x, t):
        # e = torch.randn(1, 1, 32, 32) # ruido gaussiano
        e = torch.randn_like(x)  # ruido gaussiano mismo tamaño que x
        beta_t, alpha_t = self.scheduler(t)
        # para que vaya con tensores
        beta_t = beta_t.view(-1, 1, 1, 1)
        alpha_t = alpha_t.view(-1, 1, 1, 1)
        z = torch.sqrt(alpha_t) * x + torch.sqrt(beta_t) * e
        return z, e    

In [ ]:
class Embeder(nn.Module):
    def __init__(self, num_epochs, embed_dim):
        super().__init__()
        position = torch.arange(num_epochs, device=device).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, embed_dim, 2, device=device).float() * -(math.log(10000.0) / embed_dim))
        embeddings = torch.zeros(num_epochs, embed_dim, device=device)
        embeddings[:, 0::2] = torch.sin(position * div)
        embeddings[:, 1::2] = torch.cos(position * div)
        self.embeddings = embeddings


    def forward(self, x, t):
        # t = t.long().view(-1) esto hace que t sea un tensor
        embeds = self.embeddings[t].to(x.device)
        return embeds[:, :, None, None]

In [ ]:
class DummyLayer(nn.Module):
    def __init__(self, in_channels, out_channels, norm_groups):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(norm_groups, out_channels),
            nn.ReLU()
        )
        self.res_conv = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, x, t_emb):
        out = self.conv(x)
        res = self.res_conv(out)
        return out, res


In [ ]:
'''
    La vaina es la siguiente:
    - input_channels: canales de entrada (1 para escala de grises, 3 para RGB)
    - output_channels: canales de salida (1 para escala de grises, 3 para RGB)
    - layers: capas que van a hacer vainas a la entrada, la clase aun no esta implementada -> la entrada coincide con channels 0
        -> la salida coincide con channels 1
    - embedder: objeto que va a hacer el embedding del tiempo
    - channels: 
        0. canales de salida de la primera convolucion = canales de entrada a las layers
        1. canales de salida de las layers = canales de entrada de la convolucion de salida
'''
class DiffusionModel(nn.Module):
    def __init__(self, channels:list, norm_groups:int, layers:list[nn.Module], embedder:nn.Module, input_channels=1, output_channels=1):
        super().__init__()
        self.relu = nn.ReLU()
        self.channels = channels
        
        # aparentemente esto va de hacer redes convolucionales
        # self.conv_in = nn.Conv2d(input_channels, channels[0], kernel_size=3, padding=1),

        
        self.conv_in = nn.Sequential(
            nn.Conv2d(input_channels, channels[0], kernel_size=3, padding=1),
            nn.GroupNorm(norm_groups, channels[0]),
            nn.ReLU()
        )
        
        # self.conv_out = nn.Conv2d(channels[1], output_channels, kernel_size=3, padding=1)

        self.conv_out = nn.Sequential(
            nn.GroupNorm(norm_groups, channels[1] * 2),
            nn.ReLU(),
            nn.Conv2d(2*channels[1], output_channels, kernel_size=3, padding=1),
        )
        
        self.embeder = embedder
        self.layers = layers
        
    def forward(self, x, t):
        return self.forward_withres(x, t)
    
    def forward_nores(self, x, t):
        B, C, H, W = x.shape # batch, canales, altura, anchura
        x = self.conv_in(x)
        t_emb = self.embeder(x, t)
        for layer in self.layers:
            x, r = layer(x, t_emb)
        return self.conv_out(x)
        
    def forward_withres(self, x, t):
        B, C, H, W = x.shape # batch, canales, altura, anchura
        
        x = self.conv_in(x)
        t_emb = self.embeder(x, t)
        residuals = []
        it = 0
        for layer in self.layers:
            if it < len(self.layers) //2:
                x, r = layer(x, t_emb)
                residuals.append(r)
            else:
                x = torch.concat((layer(x, t_emb)[0], residuals.pop()), dim=1)
            it += 1
        # x = self.relu(x)
        return self.conv_out(x)
    
         

In [ ]:
   
def train_old(input_size, epochs=1000, batch_size=16, lr=1e-3):
    layers = [
        DummyLayer(64, 128, 8).to(device),
        DummyLayer(128, 128, 8).to(device),
    ]
    embedder = Embeder(num_epochs=epochs, embed_dim=128)
    model = DiffusionModel(
        channels=[64, 128],
        norm_groups=8,
        layers=layers,
        embedder=embedder,
        input_channels=1,
        output_channels=1        
    ).to(device)
    
    best_loss = 1
    
    # train_ds = FashionMNIST(root=mnist_datasets_path, train=True,  download=True, transform=ToTensor())

    train_loader = DataLoader(NSynth('training'), batch_size=batch_size, shuffle=True,  pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    
    optimizer = torch.optim.Adam(diffuser.parameters(), lr=lr)
    
    mse_loss = nn.MSELoss()
        
    for epoch in range(epochs):
        for wave, _, _, _ in train_loader:
            wave = wave.to(device)
            x = stft_transform(wave)
            
            x = x.to(device)
            batch_size = x.size(0)
            t = torch.randint(0, epochs, (batch_size,), device=device)

            z, e = diffuser(x, t)
            e_pred = model(z, t)

            loss = mse_loss(e_pred, e)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            # torch.save(model.state_dict(), diff_model_path)
            print(f"Model saved at epoch {epoch} with loss {best_loss}")
    
    return model, scheduler


In [ ]:
def train_minst(input_size, epochs=1000, batch_size=16, lr=1e-3):
    layers = [
        DummyLayer(64, 128, 8).to(device),
        DummyLayer(128, 128, 8).to(device),
    ]
    embedder = Embeder(num_epochs=epochs, embed_dim=128)
    model = DiffusionModel(
        channels=[64, 128],
        norm_groups=8,
        layers=layers,
        embedder=embedder,
        input_channels=1,
        output_channels=1        
    ).to(device)
    
    best_loss = 1
    
    # train_ds = FashionMNIST(root=mnist_datasets_path, train=True,  download=True, transform=ToTensor())
    train_ds = FashionMNIST(root=mnist_datasets_path, train=True,  download=True, transform=ToTensor())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    
    optimizer = torch.optim.Adam(diffuser.parameters(), lr=lr)
    
    mse_loss = nn.MSELoss()
        
    for epoch in range(epochs):
        for wave, _, _, _ in train_loader:
            wave = wave.to(device)
            x = stft_transform(wave)
            
            x = x.to(device)
            batch_size = x.size(0)
            t = torch.randint(0, epochs, (batch_size,), device=device)

            z, e = diffuser(x, t)
            e_pred = model(z, t)

            loss = mse_loss(e_pred, e)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            torch.save(model.state_dict(), diff_mnist_model_path)
            print(f"Model saved at epoch {epoch} with loss {best_loss}")
    
    return model, scheduler

In [ ]:
   
def train(input_size, epochs=1000, batch_size=16, lr=1e-3):
    layers = [
        DummyLayer(64, 128, 8).to(device),
        DummyLayer(128, 128, 8).to(device),
    ]
    embedder = Embeder(num_epochs=epochs, embed_dim=128)
    model = DiffusionModel(
        channels=[64, 128],
        norm_groups=8,
        layers=layers,
        embedder=embedder,
        input_channels=1,
        output_channels=1        
    ).to(device)
    
    best_loss = 1
    
    # train_ds = FashionMNIST(root=mnist_datasets_path, train=True,  download=True, transform=ToTensor())

    train_loader = DataLoader(NSynth('training'), batch_size=batch_size, shuffle=True,  pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    
    optimizer = torch.optim.Adam(diffuser.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler()
    mse_loss = nn.MSELoss()
        
    for epoch in range(epochs):
        

        for wave, _, _, _ in train_loader:
            wave = wave.to(device)

            # en principio autocast hacia que sea mas eficiente en memoria pero se ve que no lo suficiente
            with torch.cuda.amp.autocast():
                x = stft_transform(wave)
                batch_size = x.size(0)

                t = torch.randint(0, epochs, (batch_size,), device=device)

                z, e = diffuser(x, t)
                e_pred = model(z, t)

                loss = mse_loss(e_pred, e)

            optimizer.zero_grad()

            # backward con escala
            scaler.scale(loss).backward()

            # step escalado
            scaler.step(optimizer)

            # actualiza escala interna
            scaler.update()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            # torch.save(model.state_dict(), diff_model_path)
            print(f"Model saved at epoch {epoch} with loss {best_loss}")
    
    return model, scheduler


In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def sample_images(model, scheduler, num_images=4, image_size=(1, 28, 28)):
    model.eval()
    
    # Empieza desde ruido puro
    x = torch.randn(num_images, *image_size, device=device)
    T = scheduler.beta.size(0)
    
    for t in reversed(range(T)):
        t_batch = torch.tensor([t]*num_images, device=device)
        # predecir el ruido
        e_pred = model(x, t_batch)
        beta_t, alpha_t = scheduler(t_batch)
        beta_t = beta_t.view(-1, 1, 1, 1)
        alpha_t = alpha_t.view(-1, 1, 1, 1)
        
        # update x según DDPM simple (un paso de denoising)
        x = (x - torch.sqrt(beta_t) * e_pred) / torch.sqrt(alpha_t)
    
    # x ahora es la imagen generada
    x = x.clamp(0, 1).cpu()
    
    # Mostrar imágenes
    fig, axes = plt.subplots(1, num_images, figsize=(num_images*2, 2))
    for i in range(num_images):
        axes[i].imshow(x[i, 0], cmap='gray')
        axes[i].axis('off')
    plt.show()
    return x


In [ ]:
model, scheduler = train_minst(input_size=(1, 28, 28), epochs=100, batch_size=256, lr=1e-3)

In [ ]:
sample_images(model, num_images=6, image_size=(1, 28, 28))

In [ ]:
# model, scheduler = train(input_size=(1, 1500, 251), epochs=10, batch_size=128, lr=1e-3)
# sample_images(model, scheduler, num_images=1, image_size=(1, 480, 480))

OutOfMemoryError: CUDA out of memory. Tried to allocate 22.98 GiB. GPU 0 has a total capacty of 24.00 GiB of which 0 bytes is free. Of the allocated memory 40.88 GiB is allocated by PyTorch, and 279.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
sample_images(model, scheduler, num_images=1, image_size=(1, 480, 480))

NameError: name 'sample_images' is not defined